# Module 01: Research Configuration

这是以 **2019 年为基期** 的完整 Module 01，可以独立使用，不需要先运行旧版。本 guide 将中文说明与完整英文代码放在一起，按 Jupyter Notebook 的 8 个 code cells 排列。每个代码格第一行都有英文注释小标题，配套 `.ipynb` 的代码与这里相同。

本版研究设置：

- 正式比较年度：2019—2024，共六年、两个分部、12 条计划分部年度记录。
- 财年配对：Amazon 2019—2024，对应 Walmart FY2020—FY2025，实际报告日期在第二模块核对。
- 销售指数：各分部以自己的 2019 分析年度销售额为基期，基期指数为 100。
- 比较阶段：2019 为疫情前参照，2020—2021 为疫情冲击阶段，2022—2024 为后续调整阶段。
- 同比增长率：仍与上年比较；2019 因未采集 2018 基线而留空，2020—2024 在数据完整时可以计算。
- 本模块产出：研究配置对象和 JSON 文件；财务数据采集、SQLite 入库与指标计算分别在后续模块实现。

每格的完整代码后都有构建思路、逐段说明与“为什么这样写”。所有函数和方法说明输入、输出、调用时机，以及相关语法。文件末尾说明从研究需求到对象设计的过程和完整调用顺序。

当前版本优先使用常规写法：普通 `def` 函数、一个简单类、`__init__`、字典、列表、`for` 循环和 `if` 判断。没有 `dataclass`、装饰器、复杂类型标注或列表推导式。保留一个配置对象，是为了落实 OOP，而不是为每个字段都建立一个类。

本次研究是 Amazon North America 与 Walmart U.S. 在疫情前后及后续调整阶段的销售增长和营业盈利能力比较。2019—2024 的阶段划分是本项目的描述性安排，2022—2024 不代表疫情已经结束。研究内容在 Cell 05，通用处理代码在 Cell 01—04，运行步骤在 Cell 06—08。前三部分的分工不依赖公司显示名称。

本项目仍有四个模块：

1. **Module 01: Research Configuration**：研究配置。
2. **Module 02: Data and Database Management**：数据获取、标准化、检查与 SQLite 存储。
3. **Module 03: Metrics and Comparative Analysis**：读取基础数据，计算并保存指标，组织比较。
4. **Module 04: Reporting and Workflow**：读取结果，组织图表、报告与整体运行。

数据库规划已写入项目的 `project_plan.md`。当前模块只预留 `database_file`，不创建数据库、不录入虚构数据。

直接打开配套 `.ipynb`，或把下方 8 段完整代码依次放入你自己的 Notebook 的 8 个 code cells。若替换旧 Module 01，请替换原有八格，重启 kernel 后从 Cell 01 到 Cell 08 顺序运行。不要把新版接在旧类定义后面运行，以免旧配置和旧对象混在一起。Notebook 内含全部函数与类，不需要导入配套 Python 文件，首次运行也不需要已有 JSON。

推荐将 Notebook 与配置文件放在项目的 `module_01` 目录。Notebook 默认导入路径为 `../data/financial_data.csv`，数据库路径为 `../data/retail_comparison.sqlite`，输出目录为 `../results`。这些只是配置约定，可以在 Cell 05 修改。若你的 Notebook 位于项目根目录，可把这三项改为 `data/financial_data.csv`、`data/retail_comparison.sqlite` 和 `results`。

Cell 编号仅指 code cells；Notebook 中附带的中文 Markdown 说明格不计入编号。修改 Cell 05 内容后，重新运行 Cell 05—08；修改函数或类后，建议重启 kernel 并运行全部 cell。


## Cell 01: Import Standard Libraries

目的：导入三个标准库。`json` 负责读写配置文件，`Path` 负责组织文件路径，`urlparse` 负责拆分来源地址。这里只导入工具，不采集或计算数据。需要 Python 3.10 或更高版本；代码无需安装第三方包，但运行 Notebook 需要已有的 Jupyter 环境。

预期结果：通常没有输出。后续 cell 会使用这些工具，导入暂时显示灰色不影响执行。


In [ ]:
# Cell 01: Import Standard Libraries

import json
from pathlib import Path
from urllib.parse import urlparse


### 构建思路：先准备工具，再定义规则

本模块需要完成三种操作：保存研究设置、处理文件路径、检查来源地址的基本格式。因此只导入与这些操作有关的工具。这一格不需要导入用于数值分析或绘图的库，因为这里还没有财务数据。

### 逐句解释

`import json` 导入 JSON 工具。后面使用 `json.dump` 把字典写入文件，使用 `json.load` 把文件内容读回字典。JSON 是文本形式的配置文件，不能把任意 Python 对象直接写进去，因此后面会先调用 `to_dict`。

`from pathlib import Path` 从路径工具库里取出 `Path`。`Path` 将路径表示为对象，可以调用 `.resolve()`、`.parent` 和 `.suffix` 等方法。路径对象之间使用 `/` 拼接文件位置，比自己拼接目录分隔符更清楚。这里的 `/` 是路径连接操作，不是数值除法。

`from urllib.parse import urlparse` 导入一个普通函数。它将来源地址拆成协议、域名和路径，方便后面检查。例如网址的 `https` 是协议，`www.sec.gov` 是域名。这个函数不会访问网络，也不会下载财报。

### 为什么运行后没有输出

导入工具只是让当前 kernel 能找到相应功能，代码没有调用 `print`。没有输出不表示运行失败。如果后续提示工具未定义，先检查这一格是否已经执行，以及 kernel 是否被重启过。

### 哪些是必须学会的，哪些是本次选择

`import` 是调用库的基础用法。这三个具体工具是本模块的实现选择；以后需要数据库时会在第二模块使用 `sqlite3`，不必在第一模块提前导入。


## Cell 02: Define Basic Validation Functions

目的：定义两个普通函数，减少重复检查。`check_text` 检查字段是不是非空文本；`check_year` 检查是不是合理的整数年份。`if` 判断不满足要求时，`raise ValueError` 给出错误说明并停止继续处理。`SUPPORTED_METRICS` 只是本项目支持的指标名称列表，尚未实现指标公式。

预期结果：通常没有输出。`def` 先定义函数，调用函数时才会执行其中的判断。


In [ ]:
# Cell 02: Define Basic Validation Functions

SUPPORTED_METRICS = ["sales_growth", "operating_margin", "sales_index"]


def check_text(value, field_name):
    """Check that a setting contains non-empty text."""
    if not isinstance(value, str) or not value.strip():
        raise ValueError(f"{field_name} must be a non-empty string.")


def check_year(value, field_name):
    """Check that a setting contains a reasonable integer year."""
    if type(value) is not int:
        raise ValueError(f"{field_name} must be an integer.")
    if value < 1900 or value > 2100:
        raise ValueError(f"{field_name} must be between 1900 and 2100.")


### 构建思路：把重复的判断写成普通函数

题目、公司名称、标识符和路径都需要检查“是否是非空文本”；分析年度和报告财年都需要检查“是否是整数年份”。如果在每个位置重复写判断，容易遗漏，也难统一修改。两个函数各解决一个明确的问题，后面可以重复调用。

### SUPPORTED_METRICS：为什么先建立允许使用的指标列表

`SUPPORTED_METRICS = [...]` 创建一个列表，存放三个允许的英文指标标识。大写名称只是常见约定，表示这份列表通常不随单次研究改变；Python 不会阻止你修改它。

`sales_growth`、`operating_margin` 和 `sales_index` 分别表示销售增长率、营业利润率和销售指数。现在只登记名字，第三模块才实现公式。以后扩展指标时，需要同时扩展支持的名称和计算代码，不能只在内容配置中填写一个新名字。

### check_text：检查文本的函数

**输入：** `value` 是要检查的值；`field_name` 是用于错误提示的字段名称。

**正常输出：** 没有显式 `return`，成功时返回 `None`。它的目的不是生成新文本，而是确认能否继续执行。

**不正常时：** 抛出 `ValueError`，停止当前执行路径，并说明哪个字段有问题。

`def check_text(value, field_name):` 定义函数。`value` 和 `field_name` 是参数，在调用时接收实际内容；定义时不会执行函数内部的检查。

`isinstance(value, str)` 判断值是否是字符串。`str` 是 Python 的字符串类型。前面的 `not` 将判断反过来，所以 `not isinstance(...)` 表示“它不是字符串”。

`value.strip()` 去掉文本两端的空白字符。空文本或只有空白的文本，处理后都是空字符串；`not value.strip()` 因而为真。这里没有把处理结果重新赋值给 `value`，所以不会修改原来的文本。

`or` 表示任意一个条件满足就报错。Python 会先判断左边；如果左边已经为真，就不再执行右边。这称为短路求值。它确保当输入是数字或 `None` 时，不会继续调用字符串才具有的 `.strip()`。

`raise ValueError(...)` 主动提出错误。`f"{field_name} ..."` 是格式化字符串：把变量的值插入大括号所在位置，使提示明确到字段名称。

**为什么不自动把数字转成文本？** 公司名称写成数字通常是配置错误。自动转换会掩盖这个错误；这里要求使用者先纠正内容，再继续运行。

例如，检查 `check_text("Amazon", "company_name")` 会正常结束；检查 `check_text("   ", "company_name")` 会报错。这两个例子用于理解函数，不需要插入正式运行流程。

### check_year：检查年份的函数

**输入：** 要检查的年份，以及对应字段名称。

**正常输出：** `None`，原年份不变。

**不正常时：** `ValueError`，说明类型或范围不正确。

`type(value) is not int` 要求值的实际类型就是整数。这意味着 `2020` 合格，`"2020"` 和 `2020.0` 不合格。这里没有使用 `isinstance(value, int)`，是因为 Python 的布尔值也会被 `isinstance` 视为整数；研究年份不应该接受 `True` 或 `False`。

`value < 1900 or value > 2100` 检查年份范围。1900—2100 是本项目选择的合理范围，不是 Python 或会计制度规定的通用限制。以后研究其他历史期间时可以调整。

先检查类型、再比较大小很重要。如果输入是文本，第一段已经报错，程序不会继续把文本与整数比较。

### 三引号、缩进和返回值

函数开头的三引号文字叫文档字符串，解释函数目的，不是业务判断。缩进表示哪些语句属于这个函数或某个 `if` 分支。不要随意删掉空格。`print` 是展示内容，`return` 是把值交给调用者，这两个检查函数主要通过“成功继续、失败报错”发挥作用。


## Cell 03: Define the Study Configuration Class

目的：用一个简单类保存研究设置，并让配置对象负责检查自己的内容。`__init__` 是创建对象时执行的初始化方法；`self` 表示当前对象，`self.title` 表示它保存的题目。`validate` 使用普通循环和判断检查设置；`get_observation_count` 计算计划观察值数；`resolve_paths` 解析路径；`to_dict` 把对象内容整理回字典。公司、分部和年份的具体内容会在 Cell 05 填写。这个类的定义需要放在一个完整 code cell 中。

预期结果：通常没有输出。此时只是定义模板，还没有创建本次研究的对象。代码较长主要是逐条数据检查，可以先读初始化与末尾三个方法，再阅读检查逻辑。


In [ ]:
# Cell 03: Define the Study Configuration Class

class StudyConfig:
    """Store and validate one study without collecting financial data."""

    def __init__(self, settings):
        if not isinstance(settings, dict):
            raise ValueError("Study settings must be a dictionary.")

        required_fields = [
            "study_id", "title", "research_question", "frequency", "currency",
            "business_units", "periods", "metrics", "index_base_year",
            "missing_previous_period", "alignment_note", "data_file",
            "results_directory",
        ]
        for field_name in required_fields:
            if field_name not in settings:
                raise ValueError(f"Missing study setting: {field_name}")

        self.study_id = settings["study_id"]
        self.title = settings["title"]
        self.research_question = settings["research_question"]
        self.frequency = settings["frequency"]
        self.currency = settings["currency"]
        self.business_units = settings["business_units"]
        self.periods = settings["periods"]
        self.metrics = settings["metrics"]
        self.index_base_year = settings["index_base_year"]
        self.missing_previous_period = settings["missing_previous_period"]
        self.alignment_note = settings["alignment_note"]
        self.data_file = settings["data_file"]
        self.results_directory = settings["results_directory"]
        self.database_file = settings.get("database_file", "../data/retail_comparison.sqlite")
        self.validate()

    def validate(self):
        """Check content before later modules use the configuration."""
        values = self.to_dict()
        text_fields = [
            "study_id", "title", "research_question", "currency",
            "alignment_note", "data_file", "results_directory", "database_file",
        ]
        for field_name in text_fields:
            check_text(values[field_name], field_name)

        if self.frequency != "annual":
            raise ValueError("Only annual studies are supported.")
        if len(self.currency) != 3 or not self.currency.isalpha() or not self.currency.isupper():
            raise ValueError("Currency must be a three-letter uppercase code.")
        if not isinstance(self.business_units, list) or len(self.business_units) < 2:
            raise ValueError("A comparison requires at least two business units.")

        unit_ids = []
        for unit in self.business_units:
            if not isinstance(unit, dict):
                raise ValueError("Each business unit must be a dictionary.")
            for field_name in ["unit_id", "company_name", "ticker", "segment_name", "currency"]:
                check_text(unit.get(field_name), field_name)
            if unit["unit_id"] in unit_ids:
                raise ValueError("Business unit IDs must be unique.")
            unit_ids.append(unit["unit_id"])
            if unit["currency"] != self.currency:
                raise ValueError("All business units must use the study currency.")

            sources = unit.get("source_urls")
            if not isinstance(sources, list) or len(sources) == 0:
                raise ValueError("Each business unit needs a list of source URLs.")
            for url in sources:
                check_text(url, "source_url")
                parsed_url = urlparse(url)
                if parsed_url.scheme not in ["http", "https"] or not parsed_url.netloc:
                    raise ValueError(f"Invalid source URL: {url}")

            notes = unit.get("scope_notes")
            if not isinstance(notes, list) or len(notes) == 0:
                raise ValueError("Each business unit needs a list of scope notes.")
            for note in notes:
                check_text(note, "scope_note")

        if not isinstance(self.periods, list) or len(self.periods) < 2:
            raise ValueError("A trend comparison requires at least two annual periods.")

        analysis_years = []
        previous_fiscal_years = {}
        for period in self.periods:
            if not isinstance(period, dict):
                raise ValueError("Each period must be a dictionary.")
            year = period.get("analysis_year")
            check_year(year, "analysis_year")
            if len(analysis_years) > 0 and year != analysis_years[-1] + 1:
                raise ValueError("Analysis years must be consecutive and ordered.")
            analysis_years.append(year)

            fiscal_years = period.get("fiscal_years")
            if not isinstance(fiscal_years, dict):
                raise ValueError("Fiscal years must be a dictionary.")
            if set(fiscal_years.keys()) != set(unit_ids):
                raise ValueError("Each period must map exactly the configured business units.")
            for unit_id in unit_ids:
                fiscal_year = fiscal_years[unit_id]
                check_year(fiscal_year, "fiscal_year")
                if unit_id in previous_fiscal_years:
                    if fiscal_year != previous_fiscal_years[unit_id] + 1:
                        raise ValueError("Each unit's fiscal years must be consecutive and ordered.")
                previous_fiscal_years[unit_id] = fiscal_year

        if not isinstance(self.metrics, list) or len(self.metrics) == 0:
            raise ValueError("Metrics must be a non-empty list.")
        used_metrics = []
        for metric in self.metrics:
            if metric not in SUPPORTED_METRICS:
                raise ValueError("Supported metrics: sales_growth, operating_margin, sales_index.")
            if metric in used_metrics:
                raise ValueError("Metric IDs must be unique.")
            used_metrics.append(metric)

        check_year(self.index_base_year, "index_base_year")
        if self.index_base_year not in analysis_years:
            raise ValueError("The index base year must be an analysis year.")
        if self.missing_previous_period != "leave_missing":
            raise ValueError("Missing growth must be left missing, not replaced with zero.")

        for file_path in [self.data_file, self.database_file, self.results_directory]:
            if Path(file_path).is_absolute():
                raise ValueError("Data, database, and result paths must be relative to the configuration file.")
        if Path(self.database_file).suffix.lower() not in [".sqlite", ".sqlite3", ".db"]:
            raise ValueError("Database files must use .sqlite, .sqlite3, or .db.")
        if Path(self.data_file) == Path(self.database_file):
            raise ValueError("The import file and database must have different paths.")

    def get_observation_count(self):
        """Return the planned number of segment-year observations."""
        return len(self.business_units) * len(self.periods)

    def resolve_paths(self, config_path):
        """Resolve paths without creating files or a database."""
        base_folder = Path(config_path).resolve().parent
        return {
            "data_file": (base_folder / self.data_file).resolve(),
            "database_file": (base_folder / self.database_file).resolve(),
            "results_directory": (base_folder / self.results_directory).resolve(),
        }

    def to_dict(self):
        """Prepare configuration content for JSON storage."""
        return {
            "study_id": self.study_id,
            "title": self.title,
            "research_question": self.research_question,
            "frequency": self.frequency,
            "currency": self.currency,
            "business_units": self.business_units,
            "periods": self.periods,
            "metrics": self.metrics,
            "index_base_year": self.index_base_year,
            "missing_previous_period": self.missing_previous_period,
            "alignment_note": self.alignment_note,
            "data_file": self.data_file,
            "results_directory": self.results_directory,
            "database_file": self.database_file,
        }


### 构建思路：让一个对象管理一份完整研究配置

一个研究配置同时拥有题目、对象、年份、指标和路径，也需要提供检查、计数与导出功能。把这些属性和相关方法放在 `StudyConfig` 中，后续模块就能接收同一种配置对象。

这就是本模块的 OOP：对象保存状态，方法处理与状态直接有关的操作。读写磁盘文件放在 Cell 04 的普通函数中，便于把内存中的配置与文件操作分开。

类定义里只写字段名称和规则；Amazon、Walmart 和具体年份在 Cell 05 填写。因此框架可以用于其他研究对象，但新增数据来源的匹配工作仍属于第二模块。

### class、对象、方法和 self

`class StudyConfig:` 定义一种对象的结构。它本身是模板；Cell 06 的 `config = StudyConfig(study_settings)` 才创建本次研究对象。

类内部的函数称为方法。每个方法的第一个参数是 `self`，表示“现在正在操作的这个对象”。调用 `config.validate()` 时，Python 会自动把 `config` 作为 `self` 传入，不需要写成 `config.validate(config)`。

`config.title` 读取一个属性；`config.get_observation_count()` 调用一个方法。后者有括号，表示要执行计算。

### 方法 1：__init__，建立对象并立即检查

**输入：** 一个名为 `settings` 的研究内容字典。

**效果：** 将字典中的内容保存成对象属性，然后调用 `validate`。初始化方法不需要返回配置对象，Python 的实例创建过程会完成这件事。

第一段 `if not isinstance(settings, dict)` 检查输入类型。这个类需要通过键读取内容，所以输入必须是字典。先检查类型，可以比后面随机出现取值错误更早地说明问题。

`required_fields` 列出必须存在的字段名。`for field_name in required_fields` 按顺序检查每个字段；`field_name not in settings` 对字典检查的是键，不是值。缺少字段时立即报错，避免继续构造不完整对象。

检查“存在”与检查“内容有效”是两件不同的事。例如 `title` 这个键存在，但值是空字符串，字段存在检查会通过，后面的文本检查会拒绝。

接下来的属性赋值逐项保存内容：

- `study_id`：研究标识，用于区分研究设置和后续结果。
- `title`：报告题目。
- `research_question`：本次研究具体要回答的问题。
- `frequency`：时间频率，当前支持年度。
- `currency`：本次统一使用的币种。
- `business_units`：要比较的分部信息列表。
- `periods`：分析年度与报告财年的配对列表。
- `metrics`：本次启用的指标列表。
- `index_base_year`：销售指数的基期。
- `missing_previous_period`：缺少上年数据时采用的处理政策。
- `alignment_note`：财年配对的说明。
- `data_file`：CSV 导入材料的位置。
- `results_directory`：报告结果目录。
- `database_file`：SQLite 文件位置。

例如 `self.title = settings["title"]` 的右侧从字典中取值，左侧将值存到当前对象。它没有把题目写死在类里。

`settings["title"]` 在键不存在时会报错；`settings.get("database_file", "../data/retail_comparison.sqlite")` 则在键不存在时使用默认值。数据库路径不是旧版配置的必填字段，默认值让旧配置也能被读取。其他必需字段不自动补全，防止形成无意义的研究配置。

最后的 `self.validate()` 自动检查新对象。如果检查失败，Cell 06 不会成功获得一个新的有效 `config`。Notebook 若此前已有旧的 `config`，失败的赋值不会自动清除旧变量，所以报错后要纠正配置并重新运行，不应继续使用旧对象。

**一个当前版本的细节：** 分部、时期和指标列表直接引用 `settings` 中的列表，没有深复制。如果你原地修改这些嵌套列表，已有对象也可能受到影响。为了让运行过程清楚，建议修改 Cell 05 后重新执行 Cell 06—08。

### 方法 2：validate，逐段检查配置规则

**输入：** 不需要额外参数，读取当前对象的属性。

**正常输出：** `None`，不改动财务数据，也不下载来源。

**不正常时：** 抛出 `ValueError`。错误提示是为了定位配置问题，不是统计检验结果。

#### 第一段：检查通用文本字段

`values = self.to_dict()` 将属性放进一个字典，方便按字段名称读取。`text_fields` 列出需要非空文本的字段。循环中的 `check_text(values[field_name], field_name)` 同时传入实际值和用于提示的字段名。

为什么 `to_dict` 写在类的后面仍可调用？Python 先完整执行类定义，之后 Cell 06 才创建实例。届时这个类的所有方法都已经定义，不要求方法按照调用顺序排列。

#### 第二段：检查频率与币种

`self.frequency != "annual"` 拒绝年度以外的频率，因为当前年份映射和指标框架只计划处理年度。若以后研究季度，需要扩展框架，不是只改这个词就可以。

币种检查要求长度为 3、全部是字母且为大写，当前内容是 `USD`。`.isalpha()` 检查是否为字母，`.isupper()` 检查是否大写。这只是编码形式检查，没有查询完整的币种标准表，也没有执行汇率转换。

#### 第三段：检查研究对象容器

`not isinstance(self.business_units, list) or len(self.business_units) < 2` 要求对象信息用列表保存，而且至少有两个。这里也利用短路求值：类型不对时不继续调用 `len`。

至少两个是比较研究的规则；代码没有限制只能恰好两个，以后可配置三家，但每个对象都必须有对应数据。

#### 第四段：检查每一个分部

`unit_ids = []` 准备一个已见过的标识符列表。`for unit in self.business_units` 每次取出一个分部字典。

先检查 `unit` 是否是字典，再对 `unit_id`、`company_name`、`ticker`、`segment_name` 和 `currency` 逐项调用 `check_text`。这里使用 `unit.get(field_name)`；如果缺少键，会得到 `None`，文本检查会给出字段错误提示。

`if unit["unit_id"] in unit_ids` 检查标识符是否已经出现；`unit_ids.append(...)` 将当前标识记入列表。唯一标识符能防止不同分部在财年映射和数据库里被混淆。公司显示名称可以修改，稳定标识符需要与数据同步。

`unit["currency"] != self.currency` 检查各对象的币种是否与研究币种一致。这里只拒绝不一致的配置；基础财报币种是否真的一致、金额单位如何转换，在第二模块核对。

#### 第五段：检查来源地址

`sources = unit.get("source_urls")` 取得来源列表。先要求它是非空列表，再逐个检查网址文本。

`parsed_url = urlparse(url)` 解析地址。`parsed_url.scheme` 是协议，必须是 `http` 或 `https`；`parsed_url.netloc` 是域名或主机部分，不能为空。

这样可以发现明显的填写错误，却不能证明链接可访问、免费或确实属于所选分部。来源内容仍需第二模块人工核对或实际读取。

#### 第六段：检查口径说明

`notes = unit.get("scope_notes")` 取得说明列表，要求至少有一条非空说明。后面的 `for note in notes` 检查每一条。

为什么说明也放入配置？计算结果需要告诉读者研究的是哪个分部、包含哪些业务。把这些文字存成内容，换研究对象时可以更新说明，而无需改报告模板。程序只检查说明是否存在，不判断说明在事实层面是否正确。

#### 第七段：检查年度列表

`periods` 必须是至少包含两个时期的列表，才能做年度趋势比较。

`analysis_years = []` 保存已经检查的分析年度。`previous_fiscal_years = {}` 保存每个分部上一次见到的财年。一个用来检查共同分析时间轴，一个用来检查各公司自己的报告时间轴。

在每轮循环中，先确认 `period` 是字典，再取出 `analysis_year`，调用 `check_year` 检查类型和范围。

`analysis_years[-1]` 表示列表最后一个已记录的年度。`year != analysis_years[-1] + 1` 要求当前年度比上一年度正好大 1。前面的 `len(analysis_years) > 0` 保证列表非空，避免第一次循环就访问不存在的最后一个元素。这里的 `and` 同样采用短路求值。

这条规则同时拒绝乱序、重复和年份缺口。它是当前连续年度设计的选择；若以后研究非连续时期，需要明确调整规则以及增长率的含义。

#### 第八段：检查财年映射是否覆盖同一批对象

`fiscal_years = period.get("fiscal_years")` 取得当前分析年度的分部财年映射。

`fiscal_years.keys()` 取得字典键。`set(...)` 将键或标识列表转换为集合，集合比较不在意排列顺序。`set(fiscal_years.keys()) != set(unit_ids)` 表示配对对象与研究对象不是同一批。

为什么要同时拒绝多出来的键和漏掉的键？只检查“包含所需对象”可能让一条旧对象映射悄悄留在配置里。要求完全一致，可以在切换研究对象时及时发现未同步修改的部分。

#### 第九段：检查每个分部的财年连续性

`for unit_id in unit_ids` 遍历所有稳定标识。`fiscal_year = fiscal_years[unit_id]` 取得对应财年，再调用 `check_year`。

`if unit_id in previous_fiscal_years` 判断该分部是否已有上一期。如果有，当前财年必须是上一财年加 1；如果没有，说明这是这个分部的第一期，不做上一期比较。

`previous_fiscal_years[unit_id] = fiscal_year` 更新记录，供下一轮使用。这个字典只存在于本次检查过程，不会写入数据库。

这可以发现把 Walmart FY2025 误填为 FY2024 造成的重复，但不能判断所配年份是否在经济意义上可比。实际日期和近似配对说明仍需保留。

#### 第十段：检查指标选择

指标必须是非空列表。`used_metrics = []` 保存已经出现的指标名称。

`metric not in SUPPORTED_METRICS` 拒绝当前不支持的指标；`metric in used_metrics` 拒绝重复指标；`used_metrics.append(metric)` 记录当前指标。

支持列表与本次选择列表分开，体现“框架提供能力、内容选择能力”。这个步骤检查名字，不执行指标计算。

#### 第十一段：检查基期与缺失政策

`check_year(self.index_base_year, "index_base_year")` 检查基期年份，随后要求它存在于分析年度列表里。基期不在研究数据中，就不能按照当前方案直接形成销售指数。

`missing_previous_period` 当前只接受 `leave_missing`，表示上期数据缺失时让增长率留空。程序不会把留空改成零，也不会自动生成上年销售额。具体计算行为在第三模块实现。

当前已把 Amazon 2019 和 Walmart FY2020 正式纳入疫情前比较，期间为 2019—2024 六年。2020 年同比可使用 2019 年销售额；2019 年同比需要 2018 基线，当前未采集，因此留空。不要把基期指数 100 或缺失增长率理解成零增长。

#### 第十二段：检查路径约定

依次检查导入文件、数据库和结果目录。`Path(file_path).is_absolute()` 判断是否为绝对路径。这里要求相对路径，让配置可以随项目文件夹迁移；运行时再由 `resolve_paths` 确定绝对位置。

`Path(self.database_file).suffix` 取得文件扩展名；`.lower()` 将字母转成小写。要求扩展名为 `.sqlite`、`.sqlite3` 或 `.db`，有助于区分数据库与 CSV。这只是命名约定，不验证现有文件内容是否真是数据库。

最后比较导入路径与数据库路径，要求二者不同，避免把同一个文件同时当成 CSV 和 SQLite。第一模块没有创建数据库，也不保证数据库文件已经存在。

### 方法 3：get_observation_count，计算计划记录数

**输入：** 当前配置对象。

**输出：** 一个整数。

`len(self.business_units)` 是研究分部数量；`len(self.periods)` 是正式比较年度数量；两者相乘得到计划分部年度记录数。本次为 2 × 6 = 12。

为什么不直接写 `return 12`？对象或年份变化后，固定数值就会过时。使用内容本身计算，才能复用框架。

这里统计的是计划数量，不是数据库里已经成功收集的记录数，也不是独立样本数量。真实数据缺失与否由第二模块检查。

### 方法 4：resolve_paths，将相对路径落实到配置所在目录

**输入：** 配置文件位置 `config_path`。

**输出：** 一个字典，三个值都是绝对路径对象。

`Path(config_path).resolve().parent` 先解析配置文件的绝对路径，再取其所在目录。`base_folder / self.data_file` 把该目录与配置中的相对路径连接；末尾的 `.resolve()` 统一处理 `..` 等相对部分。

导入文件、数据库、结果目录都按相同规则解析，避免某个模块以 Notebook 位置为基准，另一个模块却以 kernel 位置为基准。

这个方法不创建文件、不创建目录，也不连接数据库。解析出路径与该路径下已有文件是不同的事情。

例如，配置位于 `project/module_01/study.json`，数据库设置为 `../data/retail_comparison.sqlite`，解析后指向同一个项目下的 `data` 目录。若配置在项目根目录，内容中的路径应相应改为 `data/retail_comparison.sqlite`。

### 方法 5：to_dict，把对象转换为可保存内容

**输入：** 当前配置对象。

**输出：** 包含所有配置字段的普通字典。

`return {...}` 创建一个外层字典；左边的字符串是 JSON 字段名，右边的 `self.xxx` 是当前对象保存的值。逐项列出属性，读起来直接，也清楚规定哪些信息需要保存。

为什么需要它？`StudyConfig` 是自定义对象，`json.dump` 不能直接知道如何保存它。先转换成字符串、数字、列表和字典组成的结构，JSON 就可以处理。

当前方法返回新的外层字典，但内部的分部、时期和指标列表仍是同一批嵌套数据，并非深复制。它用于查看和保存，不能把返回值理解为完全隔离的备份。

本方法不检查、不打印、不写文件。调用者需要保存时由 Cell 04 完成，需要检查时由 `validate` 完成。


## Cell 04: Define Configuration File Functions

目的：用两个普通函数读写配置。`save_study_config(config, file_path)` 先检查对象，再保存 JSON；`load_study_config(file_path)` 读取 JSON，然后用 `StudyConfig(settings)` 创建对象。`with open(...)` 会在读写结束后自动关闭文件。保存文件与创建数据库是两件不同的事，这里不会创建数据库。

预期结果：通常没有输出。两个函数会在 Cell 08 被调用。


In [ ]:
# Cell 04: Define Configuration File Functions

def save_study_config(config, file_path):
    """Save settings only; do not create a financial database."""
    config.validate()
    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(config.to_dict(), file, indent=2, ensure_ascii=True)
        file.write("\n")


def load_study_config(file_path):
    """Load a JSON file and construct a validated study object."""
    with open(file_path, "r", encoding="utf-8") as file:
        settings = json.load(file)
    return StudyConfig(settings)


### 构建思路：用两个函数处理磁盘读写

配置对象负责研究内容和规则，读写文件的函数负责与磁盘交互。两个普通函数已经能清楚完成工作，因此不为这一步再创建额外的文件管理类。

### save_study_config：保存研究配置

**输入：** `config` 是配置对象；`file_path` 是写入位置，可以是字符串或 `Path` 对象。

**输出：** 正常返回 `None`，同时产生或覆盖一个 JSON 文件。不会生成财务数据库。

`config.validate()` 再检查一次当前内容。对象创建后可能被修改，保存前重新检查能减少无效配置被写入文件的情况。

`with open(file_path, "w", encoding="utf-8") as file:` 打开文件。`w` 表示写入，已存在的文件会被覆盖；`encoding="utf-8"` 明确文本编码；`as file` 给文件对象起一个当前可使用的名字。缩进块结束时文件会自动关闭，即使期间发生错误也会执行关闭过程。

`json.dump(config.to_dict(), file, indent=2, ensure_ascii=True)` 把配置字典写到文件对象。`dump` 直接写文件；它不等于 `dumps`，后者把内容转换为字符串。

`indent=2` 用两个空格缩进 JSON，方便人阅读和核对；`ensure_ascii=True` 会转义非 ASCII 字符。本项目的代码与内容本来就使用英文，这个参数并不是禁止用户输入中文的验证规则。

`file.write("\n")` 在末尾添加换行，便于文本工具显示。它不增加新配置字段。

这个函数要求文件的上级目录已经存在。它不创建目录，也没有加入自动备份机制，所以请确认保存位置后再执行。

### load_study_config：恢复研究配置

**输入：** 已保存 JSON 文件的位置。

**输出：** 一个新的、通过验证的 `StudyConfig` 对象。

`with open(file_path, "r", encoding="utf-8") as file:` 以读取方式打开文件；`r` 不会覆盖原文件。

`settings = json.load(file)` 把 JSON 内容解析为 Python 数据结构。配置顶层应该是字典，内部包含列表、数字和文本。如果文件不是有效 JSON，解析会报错；如果文件不存在，打开文件会报错。

`return StudyConfig(settings)` 用读取到的内容建立对象。它会调用 `__init__`，随后自动调用 `validate`。这样手工填写的配置和从磁盘读取的配置使用同一套规则，不维护两份不同的检查代码。

为什么一定要 `return`？后面需要使用读取结果；只有把对象返回，`reloaded_config = load_study_config(...)` 才能拿到这个对象。单纯打印文件内容不能替代返回对象。

### 为什么研究配置保存成 JSON，财务数据计划保存成数据库

研究配置体量小，主要由少量文字和选项组成，JSON 便于直接查看与修改。财务记录需要按对象、日期和指标查询、检查重复和复用，数据库职责放在第二模块。两者是不同内容，不能把生成 JSON 误认为已经建立财务数据库。


## Cell 05: Set the Research Content

目的：填写本次研究内容。`study_settings` 是普通字典，键表示字段名称，值表示实际内容；列表保存多个研究对象和年度。换对象、年份、指标或路径时，优先修改这一格，前四格通常保持不变。当前是两个分部、六组年度配对，销售指数以 2019 年为基期；2019 为疫情前参照，2020—2021 为疫情冲击阶段，2022—2024 为后续调整阶段。来源链接只是数据入口，不是已采集的数据。`database_file` 预留数据库路径，第二模块再实现数据库。

预期结果：通常没有输出。这格只填写内容，下一格才将内容交给配置对象进行检查。


In [ ]:
# Cell 05: Set the Research Content

study_settings = {'study_id': 'amazon_walmart_segment_comparison',
 'title': 'Sales Growth and Operating Profitability of Internet-Origin and Store-Origin Retailers: '
          'Amazon North America and Walmart U.S. Around the COVID-19 Pandemic, 2019-2024',
 'research_question': 'Relative to a 2019 pre-pandemic baseline, how did sales growth and '
                      'operating margins differ between Amazon North America and Walmart U.S. '
                      'during 2020-2021 and the subsequent 2022-2024 adjustment period? This is a '
                      'descriptive comparison, not an estimate of the causal effect of COVID-19.',
 'frequency': 'annual',
 'currency': 'USD',
 'business_units': [{'unit_id': 'amazon_north_america',
                     'company_name': 'Amazon.com, Inc.',
                     'ticker': 'AMZN',
                     'segment_name': 'North America',
                     'currency': 'USD',
                     'source_urls': ['https://ir.aboutamazon.com/annual-reports-proxies-and-shareholder-letters/default.aspx',
                                     'https://www.sec.gov/Archives/edgar/data/1018724/000101872422000005/amzn-20211231.htm',
                                     'https://www.sec.gov/Archives/edgar/data/1018724/000101872425000004/amzn-20241231.htm'],
                     'scope_notes': ['Use North America segment net sales and operating income, '
                                     'not consolidated totals.',
                                     'The segment excludes AWS but includes advertising, '
                                     'subscription services, and third-party seller services.',
                                     "The segment's geographical scope and revenue recognition "
                                     'differ from Walmart U.S.']},
                    {'unit_id': 'walmart_us',
                     'company_name': 'Walmart Inc.',
                     'ticker': 'WMT',
                     'segment_name': 'Walmart U.S.',
                     'currency': 'USD',
                     'source_urls': ['https://stock.walmart.com/financial-information/annual-reports',
                                     'https://www.sec.gov/Archives/edgar/data/104169/000010416922000012/wmt-20220131.htm',
                                     'https://stock.walmart.com/sec-filings/all-sec-filings/content/0000104169-25-000021/wmt-20250131.htm/wmtexhibit991fy2510-k.htm'],
                     'scope_notes': ['Use Walmart U.S. segment net sales and operating income, not '
                                     'consolidated totals.',
                                     "Exclude Sam's Club and Walmart International from the "
                                     'selected segment data.',
                                     'Walmart fiscal years end on January 31 of the labeled '
                                     'year.']}],
 'periods': [{'analysis_year': 2019,
              'fiscal_years': {'amazon_north_america': 2019, 'walmart_us': 2020},
              'comparison_stage': 'Pre-pandemic baseline'},
             {'analysis_year': 2020,
              'fiscal_years': {'amazon_north_america': 2020, 'walmart_us': 2021},
              'comparison_stage': 'Pandemic disruption'},
             {'analysis_year': 2021,
              'fiscal_years': {'amazon_north_america': 2021, 'walmart_us': 2022},
              'comparison_stage': 'Pandemic disruption'},
             {'analysis_year': 2022,
              'fiscal_years': {'amazon_north_america': 2022, 'walmart_us': 2023},
              'comparison_stage': 'Subsequent adjustment'},
             {'analysis_year': 2023,
              'fiscal_years': {'amazon_north_america': 2023, 'walmart_us': 2024},
              'comparison_stage': 'Subsequent adjustment'},
             {'analysis_year': 2024,
              'fiscal_years': {'amazon_north_america': 2024, 'walmart_us': 2025},
              'comparison_stage': 'Subsequent adjustment'}],
 'metrics': ['sales_growth', 'operating_margin', 'sales_index'],
 'index_base_year': 2019,
 'missing_previous_period': 'leave_missing',
 'alignment_note': 'Amazon years ending December 31 are paired with Walmart fiscal years ending '
                   'January 31 of the following year. These are approximately corresponding '
                   'periods, not identical periods. Actual start and end dates must be recorded '
                   'and checked in the data module.',
 'data_file': '../data/financial_data.csv',
 'results_directory': '../results',
 'database_file': '../data/retail_comparison.sqlite'}


### 构建思路：把会变的内容集中在一个字典里

这一格不定义新函数，而是为已有框架提供具体内容。`study_settings = {...}` 将一个字典赋给变量，之后通过这个变量创建配置对象。

大括号 `{}` 表示字典，键值之间使用冒号；方括号 `[]` 表示列表，元素用逗号分隔。`business_units` 是包含多个字典的列表，`periods` 也是列表；每个时期里的 `fiscal_years` 又是一个字典。不同的括号组合对应不同层次，不要仅凭视觉随意交换。

### 第一段：研究标识、题目、问题、频率和币种

`study_id` 用于内部标识这份研究。题目修改后不一定需要换标识；如果开始一个新的比较项目，应使用能区分研究的新标识。以后指标结果还需要结合配置签名，防止相同研究标识下的不同设置混淆。

`title` 是用于报告展示的英文题目。`research_question` 把研究限定为两个分部在研究期内的增长和利润率差异，避免仅列出数据却没有问题。

`frequency: annual` 声明年度研究；`currency: USD` 声明本次统一币种。它们目前属于内容声明，不是从财报自动识别出来的事实。

### 第二段：两个分部字典

每个对象有相同的键，通用循环就能用相同方式读取它们：

- `unit_id`：数据匹配标识，必须唯一并与财年映射、数据库记录一致。
- `company_name`：展示用公司名称。
- `ticker`：股票代码，当前只是保存识别信息，不会因此自动下载数据。
- `segment_name`：明确研究分部，防止误取集团合并数字。
- `currency`：该分部数据预期使用的币种。
- `source_urls`：官方入口与选定报告示例；当前来源覆盖六年，具体分部表与采用版本在数据流程 guide 中记录。
- `scope_notes`：说明业务范围和比较限制，后续可以传给报告模块。

Amazon 使用 `amazon_north_america`；Walmart 使用 `walmart_us`。下划线只是可读命名方式，不是 Python 自动识别公司或分部的特殊语法。

本次选择分部数据，排除 Amazon 的 AWS 与 Walmart 的其他主要分部，但 Amazon 北美仍包含广告、订阅等业务，两个对象的收入确认和地域范围仍有差异。配置保留这些限制，是为了在结果解释时记得它们，而不是声称差异已经消失。

### 第三段：六组年度配对和疫情比较阶段

每组 `analysis_year` 表示报告图表上采用的共同时间标签；`fiscal_years` 指定两个分部实际要读取的报告财年。

例如共同分析年度 2024，对应 Amazon 2024 与 Walmart FY2025。财年映射的键必须使用前面的 `unit_id`，不能一处写完整公司名称，另一处写股票代码。

配置明确列出每一组配对，而没有在通用框架里写“只要是 Walmart 就把年份加 1”。这样换公司、换财年安排时，修改的是内容，而不是计算代码。

`comparison_stage` 是本项目新增的可选内容字段，放在每个年度字典内。2019 对应 `Pre-pandemic baseline`，2020—2021 对应 `Pandemic disruption`，2022—2024 对应 `Subsequent adjustment`。标签由内容决定，通用类不判断特定公司的疫情阶段；它随 `periods` 一起保存在对象和 JSON 中。当前 `validate` 检查年度映射，不额外校验这个可选标签，后续分组前要检查标签是否完整。

2019 既用于判断疫情前的营业利润率水平，也用于令各分部销售指数等于 100。后续会比较指数相对 100 的变化和营业利润率相对 2019 的百分点变化。这里只描述阶段差异，不能据此识别疫情因果效应；2022—2024 的后续调整阶段也可能仍含疫情影响。

`analysis_year` 是分析标签，不会改变原始数据的日期。第二模块需要另外记录真实的开始与结束日期，不能把所有配对自动认为覆盖完全相同的日历期间。

### 第四段：指标、基期和缺失政策

`metrics` 从框架支持的三个指标中选择本次使用的项目。换指标选择可以改这里；增加新计算能力仍要修改第三模块的计算实现。

`index_base_year: 2019` 表示销售指数以疫情前 2019 年为基期。后续公式为当期销售额除以基期销售额，再乘 100，因此基期通常为 100。当前只是声明，尚未计算。

`missing_previous_period: leave_missing` 表示缺少上期销售额时增长率留空，不填零。当前六年数据能支持 2020—2024 的同比，2019 首年同比仍留空，因为没有 2018 基线。同比增长率始终与上年比较，不是统一与 2019 比较。

### 第五段：说明和三个路径

`alignment_note` 保存两家公司期间近似配对的解释，后面预览和报告都可以读取。

`data_file` 是 CSV 导入材料；`database_file` 是计划的 SQLite 数据文件；`results_directory` 是图表和报告目录。设置这些字符串并不创建文件。

默认的 `../` 表示相对于配置所在目录向上一层。若 JSON 位于 `module_01`，这会定位到项目根目录再进入 `data` 或 `results`。你的截图里 Notebook 位于项目根目录；如果继续采用这个布局，可将三项改为 `data/financial_data.csv`、`data/retail_comparison.sqlite` 和 `results`，让文件保存在当前项目内部。

### 为什么这是内容与框架分离

Cell 03 只知道如何读取“研究对象列表”“年度列表”“指标列表”，不会按 Amazon 或 Walmart 名称分支判断。Cell 05 提供本次内容。已有标准数据的对象可以通过配置切换；新公司的首次接入还需提供对应数据与来源映射。

运行这一格只创建普通字典，不触发配置验证、不访问网络、不生成数据库。验证在下一格执行。


## Cell 06: Create and Validate the Configuration Object

目的：用本次内容创建配置对象。调用 `StudyConfig(study_settings)` 会执行初始化方法，保存属性并调用检查。普通字典负责提供内容，类负责统一处理规则，这是本模块的 OOP 与内容分离设计。配置通过检查不代表真实财务数据已经收集或验证。

预期结果：应输出 `Configuration is valid.`。若有配置错误，根据英文提示修改 Cell 05，然后重新运行 Cell 05—08。


In [ ]:
# Cell 06: Create and Validate the Configuration Object

config = StudyConfig(study_settings)
print("Configuration is valid.")


### 构建思路：让内容通过统一入口进入框架

前面的 cell 分别准备了工具、函数、类和内容。现在把它们接起来：用内容字典创建研究对象，再确认创建成功。

### 第一行：创建对象并保存到变量

`config = StudyConfig(study_settings)` 先执行右侧，再将结果赋给左侧。

右侧会分配实例并调用 `__init__`，初始化方法逐项保存内容，末尾调用 `self.validate()`。检查过程中有错误，会立即停止这一行；成功后，左侧变量 `config` 指向新对象。

为什么后续不继续到处读取原字典？对象把研究信息和相关规则一起提供给其他模块，后续可以统一使用 `config.title`、`config.periods` 和对象方法。

### 第二行：确认配置检查已通过

`print("Configuration is valid.")` 只有在第一行成功结束后执行。这个提示只说明配置规则通过，不说明年报已下载、数据已入库，也不说明统计结论成立。

`config` 是变量名，不是固定关键字，但整个 Notebook 的后续 cell 都按这个名称引用它。为了避免混乱，初学阶段不建议随意改名。

### 这一格失败时的处理顺序

先看报错字段，再改 Cell 05，重新运行 Cell 05 和 Cell 06。通过后再运行 Cell 07—08。如果修改的是类或函数定义，重启 kernel 并运行全部 cell，避免旧对象还关联着旧定义。


## Cell 07: Review the Study Scope and Fiscal Mapping

目的：打印研究范围，人工确认读到的内容是否正确。通过 `config.title` 读取对象属性，通过 `config.get_observation_count()` 调用对象方法。研究对象和年份采用普通 `for` 循环逐项打印，没有列表推导式。财年配对只是近似对应；第二模块还需核对实际日期。Amazon 北美分部排除 AWS，但保留广告、订阅与第三方卖家服务等口径差异。

预期结果：应看到两个业务分部、12 个计划观察值、2019 基期和各年度比较阶段，以及 Amazon 2019—2024 对应 Walmart FY2020—FY2025。


In [ ]:
# Cell 07: Review the Study Scope and Fiscal Mapping

print(f"Study: {config.title}")
print(f"Research question: {config.research_question}")
print(f"Currency: {config.currency}")
print(f"Business units: {len(config.business_units)}")
print(f"Planned observations: {config.get_observation_count()}")
print(f"Metrics: {', '.join(config.metrics)}")
print(f"Index base year: {config.index_base_year}")

print("\nBusiness unit scope:")
for unit in config.business_units:
    print(f"{unit['company_name']} / {unit['segment_name']}")
    for note in unit["scope_notes"]:
        print(f"  - {note}")

print("\nFiscal period mapping:")
for period in config.periods:
    print(f"Analysis year: {period['analysis_year']}")
    print(f"  Comparison stage: {period.get('comparison_stage', 'Not specified')}")
    for unit in config.business_units:
        unit_id = unit["unit_id"]
        fiscal_year = period["fiscal_years"][unit_id]
        print(f"  {unit['company_name']}: FY{fiscal_year}")

print(f"\nAlignment note: {config.alignment_note}")
print("Financial data has not been loaded or calculated.")


### 构建思路：在保存前，让人看见程序理解的研究范围

自动验证能发现重复年份，却未必知道你是否选错了分部或研究题目。因此增加预览，人工检查“代码实际使用的内容”与“我打算研究的内容”是否一致。

### 第一段：打印研究摘要

每个 `print(f"...{...}")` 将说明文字与对象属性组合显示。`config.title`、`config.research_question`、`config.currency` 都是前面初始化保存的值。

`len(config.business_units)` 数分部数量。`config.get_observation_count()` 调用方法，返回分部数量乘正式年度数量。本次预期是 2 个分部、12 条计划记录。

`', '.join(config.metrics)` 将指标列表中的文本用逗号和空格连接成一行。它只改善显示，不修改原列表。`config.index_base_year` 显示销售指数基期，供人确认。

### 第二段：打印每个研究对象及口径

`print("\nBusiness unit scope:")` 中的 `\n` 表示换行，让不同输出部分分开。

外层 `for unit in config.business_units` 每次取出一个分部字典，使用 `unit['company_name']` 与 `unit['segment_name']` 展示身份。

内层 `for note in unit["scope_notes"]` 再逐条显示该分部的说明。内层循环全部结束后，外层才继续处理下一个分部，这就是嵌套循环。

格式化字符串外面使用双引号、取字典键的内部使用单引号，是为了清楚区分字符串边界。公司信息来自内容，不是重新在预览代码里填写公司名称。

### 第三段：打印分析年度与财年配对

`period.get("comparison_stage", "Not specified")` 读取该年度的阶段标签。`.get` 的第二个参数是标签缺失时的显示文字，所以其他研究不填写疫情标签也能使用同一段预览代码。它只是显示规则，不负责验证分组内容。

外层循环 `for period in config.periods` 遍历六个正式分析年度。每轮先打印共同的 `analysis_year`，再遍历全部研究对象。

`unit_id = unit["unit_id"]` 取得稳定标识；`fiscal_year = period["fiscal_years"][unit_id]` 先进入当前时期的财年字典，再根据标识取出该分部的报告财年。

这个两步取值是连接“对象信息”与“期间信息”的关键。显示名称可以变化，只要稳定标识与数据同步，就仍可取到正确配对。

`FY` 只是输出中的财年文字前缀，不参与年份计算。你应看到分析年度 2019 对应 Amazon FY2019 和 Walmart FY2020，直到 2024 对应 FY2024 和 FY2025。

### 第四段：显示配对限制和当前进度

`config.alignment_note` 提醒近似期间仍有差异。最后一行提醒当前没有读取或计算财务数据，避免把配置摘要当成财务结果。

这一格的输出属于运行中的检查信息，不是保存在 Notebook 外的数据库。保存 Notebook 可以保存已有输出，但后续正式分析仍应读取配置与数据库，而不是解析这些打印文字。


## Cell 08: Save and Reload the Configuration

目的：调用普通函数保存配置，并确认可以重新读取。`Path.cwd()` 表示当前 kernel 工作目录，不保证是 Notebook 所在目录；运行前可执行 `print(Path.cwd())` 确认位置。下面会写入或覆盖该目录的 `study.json`，如果位置不合适，请修改 `CONFIG_PATH`。最后分别打印 CSV 导入文件、数据库和结果目录的路径。

预期结果：应输出保存和读取成功，以及三个绝对路径。默认相对路径以 JSON 所在目录为基准；当前只生成 JSON，不生成数据库或财务数据。


In [ ]:
# Cell 08: Save and Reload the Configuration

CONFIG_PATH = Path.cwd() / "study.json"
print(f"Config file: {CONFIG_PATH.resolve()}")

save_study_config(config, CONFIG_PATH)
reloaded_config = load_study_config(CONFIG_PATH)

if reloaded_config.to_dict() != config.to_dict():
    raise ValueError("The saved configuration does not match the original.")

paths = reloaded_config.resolve_paths(CONFIG_PATH)
print("Configuration saved and reloaded successfully.")
print(f"Import file: {paths['data_file']}")
print(f"Database file: {paths['database_file']}")
print(f"Results directory: {paths['results_directory']}")
print("No financial data file, database, or results directory was created.")


### 构建思路：把内存设置保存下来，并检验恢复结果

前七格建立了内存中的配置对象，重启 kernel 后对象会消失。这一格将内容写入 JSON，再读取回来比较，确认后续模块能够恢复同一配置。

### 第一段：确定配置保存位置

`CONFIG_PATH = Path.cwd() / "study.json"` 取当前 kernel 工作目录，再拼接文件名。

`CONFIG_PATH` 用大写只是名称约定，不代表值不可修改。`.cwd()` 取得的是运行工作目录，不一定是你在编辑器中打开的 Notebook 目录。运行前可在自己的临时 cell 查看 `print(Path.cwd())`，确认位置后再执行正式保存。

`print(f"Config file: {CONFIG_PATH.resolve()}")` 显示绝对位置。这个显示语句不会暂停等待确认，下面的保存仍会马上执行，因此需要在运行整格前检查或调整路径。

### 第二段：保存，再恢复成另一个对象

`save_study_config(config, CONFIG_PATH)` 调用 Cell 04 的保存函数，覆盖该位置的 JSON 文件。其他财务文件不会在这一行生成。

`reloaded_config = load_study_config(CONFIG_PATH)` 调用读取函数，得到一个新的对象。这个对象读取自磁盘，不是仅给原对象换一个变量名。

为什么立刻读回？可以检查“写进去的内容”和“程序重新读到的内容”是否一致，这为跨 Notebook 复用配置做准备。

### 第三段：比较内容是否一致

`reloaded_config.to_dict() != config.to_dict()` 比较两个普通字典的内容。因为当前类没有自定义对象相等规则，直接比较两个对象不能表达这里想检查的“所有设置内容一致”，所以先转换为字典。

如果内容不一致，`raise ValueError(...)` 停止执行。比较一致时继续。这个检查是配置读写核对，不是对真实财务数据的真实性检查。

### 第四段：解析后续文件位置

`paths = reloaded_config.resolve_paths(CONFIG_PATH)` 以 JSON 所在目录为基准解析三条相对路径，返回一个字典。

`paths['data_file']` 是导入材料位置，`paths['database_file']` 是数据库位置，`paths['results_directory']` 是结果目录。分别打印它们，方便确认是否都在预期项目内。

最后的提示说明这一格只保存 JSON，没有创建数据库、财务 CSV 或结果目录。数据库创建和数据导入将在第二模块进行。

### 本模块结束后留下什么

当前 kernel 中留下 `config`、`reloaded_config` 和 `paths` 等变量；磁盘上留下 `study.json`。如果重启 kernel，内存变量会消失，但 JSON 文件仍在。后续 Notebook 定义或导入同一配置类与读取函数后，可以通过文件恢复设置。


## 从需求到代码：完整构建过程

**第一步，先确定可变内容。** 公司、分部、期间、指标、基期、来源和文件位置会随研究变化，把它们整理成字典字段。

**第二步，确定需要保持一致的规则。** 必需字段不能缺，比较对象不能重复，年度要连续，财年映射要覆盖所有对象，指标要受支持。这些属于框架，不属于某家公司的具体内容。

**第三步，找重复操作。** 文本和年份检查会反复出现，拆成两个小函数。配置文件保存与加载各有明确职责，拆成两个普通函数。

**第四步，决定哪里值得建立对象。** 一份研究设置具有共同生命周期：初始化、验证、供后续模块使用、保存与恢复。用一个 `StudyConfig` 类管理这些状态与方法；分部和期间暂时用普通字典表示，保持首版易读。

**第五步，明确执行顺序。** 先定义工具和函数，再定义类，再填内容，最后创建对象、预览和保存。类定义不是配置实例；函数定义也不是函数执行。

**第六步，验证复用范围。** 当前框架可以切换符合相同配置结构的研究对象和年度，但没有自动识别任意财报的能力。数据适配、单位核对和数据库管理在第二模块完成，指标公式在第三模块实现，模板与运行在第四模块组织。

### 本次创建对象时的调用顺序

Cell 06 调用 `StudyConfig(study_settings)` → Python 执行 `__init__` → 保存属性 → 调用 `validate` → `validate` 调用 `to_dict`、`check_text`、`check_year` 等工具 → 全部通过 → 得到 `config`。

### 本次保存与恢复时的调用顺序

Cell 08 调用 `save_study_config` → 再次调用 `config.validate` → 转成字典 → 写 JSON → 调用 `load_study_config` → 读 JSON → 创建另一个 `StudyConfig` 并验证 → 返回 `reloaded_config` → 比较两个对象的配置内容 → 解析路径。

### 如何判断自己理解了，而不只是复制了代码

你应该能够说明：研究内容在 Cell 05；校验规则在函数和类里；`config.title` 是属性而 `config.get_observation_count()` 是方法；保存 JSON 与建立数据库是不同步骤；年度映射使用稳定标识而不是显示名称；基期和研究年份调整后需要重新建立并检查配置对象。

可以先在纸上推演一个错误，例如将最后一个 Walmart 财年重复填写为 FY2024，找出是 `validate` 的哪个循环发现错误。这样练习不会改变正式数据，也不用添加复杂代码。

### 为什么本次没有加入更多“高级 OOP”

这个项目当前只有少量配置和两个研究对象。普通类与函数已能表达职责，继承、装饰器、抽象基类、ORM 等并不会自动让研究更科学。以后确实需要扩展数据来源时，再增加简单的数据接口。OOP 的价值在于职责清楚和可复用，而不是类的数量。

## 可选说明：旧 Python 脚本中的 main

Notebook 的八格不使用 `main`，也不需要 `argparse` 或 `__file__`。此前保留的 `study_config.py` 还可以作为终端脚本运行，以下只解释它额外的入口函数。

`main()` 创建 `ArgumentParser`，用 `add_argument("--config", ...)` 定义可选配置路径，`parse_args()` 读取终端参数。默认位置通过 `Path(__file__).with_name("study.json")` 找到脚本旁的 JSON。`__file__` 是脚本文件位置，Notebook 通常不能按相同方式使用它。

`try` 中调用 `load_study_config`；`except (OSError, ValueError)` 捕捉文件访问或配置错误，再通过 `parser.error` 显示命令行错误并结束运行。成功时打印题目与计划观察值数。

末尾的 `if __name__ == "__main__": main()` 表示只有直接运行脚本时才调用入口；将脚本作为模块导入时，不自动执行这个入口。以上都是终端版本的辅助代码，不需要复制到你的 Notebook。


## 本模块完成了什么

通过一个普通 `StudyConfig` 类，把研究设置与检查、计数、路径解析和转换方法组合起来。`config` 是本次研究的对象；读写 JSON 由两个普通函数完成。

`unit_id` 是稳定的数据标识，名称只是显示内容。换研究对象时，需要同步修改对象信息、稳定标识、财年映射、来源、口径和对应数据，不能只换名称就复用旧公司的数字。

六年的数据提供 12 个分部年度观察值。2019 是正式比较中的疫情前参照年，也是销售指数基期。销售指数和营业利润率覆盖六年，2020—2024 增长率可以使用上年数据；2019 增长率因未收集 2018 基线而留空。第三模块保持三个指标，并按配置中的阶段标签组织比较；第四模块标示阶段及相对 2019 的变化。只有一个疫情前年份，结果属于描述性比较，不能由此识别疫情的因果效应。

第二模块将使用 SQLite：通过检查的基础记录先入库，第三模块再计算并保存指标。原始财务记录与指标结果分开保存，保留来源、币种、单位和实际报告日期。CSV 保留为导入及核对材料。数据库方案见 `../project_plan.md`。

代码检查配置结构、URL 格式和内部规则，不联网核验财报内容，也不证明来源网站始终可访问。当前输出只有 `config` 和 JSON 配置文件，不包含真实财务数据或研究结论。

后续模块会保持这一写法：能够用普通函数表达的步骤就用普通函数；需要保存状态、提供稳定接口的职责才建立简单类。每个模块都有英文标题、英文注释 cell 标题、中文说明和完整英文代码。
